# Crop Recommendation — EDA e Pipeline de Pré-processamento

**Parte 1 do desafio técnico.** Este notebook entrega:

- Carregamento via `kagglehub.load_dataset` (com fallback para CSV local).
- Diagnóstico de qualidade (nulos, duplicatas, plausibilidade física).
- Distribuição da variável-alvo e por feature.
- Boxplots e violins por cultura para as 7 features.
- Mapa de correlação.
- Heatmap de assinatura nutricional/climática por cultura.
- Pairplot amostrado.
- Construção, fit e round-trip do pipeline serializado.

Toda lógica reutilizável fica em `src/crop_reco/`. O notebook é a **narrativa**;
os módulos são o **produto**. Os gráficos são salvos em `reports/figures/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)

from crop_reco import load_dataset, split_xy, build_pipeline, run_eda
from crop_reco import eda as eda_mod
from crop_reco.config import FIGURES_DIR, PIPELINE_PATH, NUMERIC_FEATURES, FINAL_FEATURES

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregamento via kagglehub

In [ ]:
df = load_dataset()
print(f"shape: {df.shape}")
df.head()

## 2. Diagnóstico de qualidade

In [ ]:
df.info()

In [ ]:
diag = pd.DataFrame({
    "n_nulos": df.isna().sum(),
    "n_unicos": df.nunique(),
    "dtype": df.dtypes.astype(str),
})
diag.loc["__duplicatas__"] = [df.duplicated().sum(), "-", "-"]
diag

**Observação:** dataset cirurgicamente limpo — 0 nulos, 0 duplicatas, todos os
tipos numéricos esperados. Não é necessária imputação.

### Balanço de classes

In [ ]:
contagem = df["label"].value_counts()
print(f"Classes: {len(contagem)}  |  min: {contagem.min()}  |  max: {contagem.max()}")

eda_mod.plot_class_distribution(df, FIGURES_DIR / "01_class_distribution.png")
from IPython.display import Image
Image(filename=str(FIGURES_DIR / "01_class_distribution.png"))

**Observação:** 22 classes × 100 amostras cada — **balanço perfeito**. Não
precisamos de oversampling, SMOTE ou pesos por classe.

### Plausibilidade física vs. outliers IQR

In [ ]:
print(">>> Validade física (sensor sano?):")
display(eda_mod.physical_validity_report(df))

In [ ]:
print(">>> Outliers IQR (estatísticos):")
eda_mod.summarize_outliers_iqr(df)

**Decisão: não remover nenhum 'outlier'.** Dois argumentos:

1. **Validade física:** 0 violações. Todos os valores estão dentro de limites
   químico-biológicos plausíveis (pH ∈ [3.5, 9.9], N/P/K ≥ 0, temperatura
   ∈ [9°C, 44°C]). Não há indício de sensor corrompido.

2. **IQR é global, fisiologia é por cultura.** Os 9% de "outliers" em K são
   o efeito de 22 distribuições muito distintas empilhadas. Uvas e maçãs
   exigem K alto por natureza — esses valores extremos **são o sinal** que
   distingue essas classes. Remover seria destruir informação preditiva.

Mesmo assim, o pipeline inclui um `PhysicalBoundsClipper` para proteger
**inferência em produção** contra leituras absurdas de sensor (ex.: pH = 25).

## 3. Distribuições

In [ ]:
eda_mod.plot_feature_distributions(df, FIGURES_DIR / "02_feature_distributions.png")
Image(filename=str(FIGURES_DIR / "02_feature_distributions.png"))

- **N/P/K**: distribuições amplas, bimodais em alguns casos (assinatura das
  diferentes famílias de culturas).
- **temperature, humidity, ph**: aproximadamente unimodais, próximas de normais.
- **rainfall**: cauda longa à direita (rice puxa para cima).

Distribuições unimodais e bem-comportadas confirmam que **`StandardScaler`
é mais adequado que `RobustScaler` ou `MinMaxScaler`** (este último sofreria
com a cauda de `rainfall`).

## 4. Comportamento por cultura

In [ ]:
eda_mod.plot_boxplots_by_class(df, FIGURES_DIR / "03_boxplots_by_class.png")
Image(filename=str(FIGURES_DIR / "03_boxplots_by_class.png"))

In [ ]:
eda_mod.plot_violin_by_class(df, FIGURES_DIR / "04_violins_by_class.png", features=["N","P","K","ph"])
Image(filename=str(FIGURES_DIR / "04_violins_by_class.png"))

**Leitura agronômica:**

- **N**: `cotton`, `coffee`, `banana`, `rice` demandam N alto; leguminosas
  (`lentil`, `kidneybeans`, `pigeonpeas`) ficam baixas — coerente com a
  capacidade dessas plantas de fixar nitrogênio atmosférico.
- **P e K**: `apple` e `grapes` saltam dos demais — frutíferas perenes
  exigem muito potássio e fósforo para frutificação.
- **pH**: faixa estreita por cultura (`coconut` e `banana` em solos ácidos,
  `chickpea` e `mothbeans` em solos neutros/alcalinos). Esta é uma feature
  altamente discriminativa.
- **rainfall**: `rice` precisa de muito mais que qualquer outra cultura
  (caule submerso) — feature isoladamente quase suficiente para distingui-la.

## 5. Correlações

In [ ]:
eda_mod.plot_correlation_heatmap(df, FIGURES_DIR / "05_correlation_heatmap.png")
Image(filename=str(FIGURES_DIR / "05_correlation_heatmap.png"))

**Análise:**

- Único par relevante: **P ↔ K (ρ = 0.74)**. Faz sentido agronomicamente —
  culturas frutíferas com alta exigência de P também demandam K.
- Apesar da correlação alta, **mantenho ambas**. Razões: (i) com apenas 7
  features, custo computacional desprezível; (ii) modelos lineares são
  pouco prejudicados em ρ < 0.9; (iii) modelos baseados em árvore são
  robustos a multicolinearidade; (iv) manter preserva interpretabilidade
  nutricional.
- Demais pares: |ρ| < 0.25 → features essencialmente independentes.

## 6. Assinatura nutricional/climática por cultura

In [ ]:
eda_mod.plot_class_signature(df, FIGURES_DIR / "06_class_signature.png")
Image(filename=str(FIGURES_DIR / "06_class_signature.png"))

Heatmap dos z-scores das **médias por cultura** (linha = cultura, coluna =
feature). Vermelho intenso ↔ feature muito alta para aquela cultura
relativa às outras; azul ↔ muito baixa.

**Padrões nítidos:**

- `apple` e `grapes`: P e K muito altos (z > 2).
- `rice`: rainfall extremo (z = 2.6).
- `papaya`: temperatura extrema.
- `chickpea`: clima frio/seco (humidity z = −2.4).
- `kidneybeans`: clima frio e pH ácido.

Essa separação tão clara entre classes é por que **classificadores simples
têm performance >95% nesse dataset**. Boa notícia para a etapa de modelagem.

## 7. Pairplot amostrado

In [ ]:
eda_mod.plot_pairplot_sample(df, FIGURES_DIR / "07_pairplot_sample.png", classes=["rice","coffee","grapes","watermelon"])
Image(filename=str(FIGURES_DIR / "07_pairplot_sample.png"))

Mostra **4 culturas** selecionadas (escolhidas para representar contraste:
arroz/café/uvas/melancia). Observe a clara separação em vários eixos —
`grapes` se destaca em P-K, `rice` em rainfall, `coffee` e `watermelon`
ocupam regiões intermediárias mas distintas. A separabilidade no espaço
original já é alta — feature engineering ajuda mas não é estritamente
necessária para acurácia.

## 8. Estratégia de pré-processamento

A pipeline aplica 4 etapas, em ordem:

1. **`FeatureValidator`** — schema enforcement em prod. Falha cedo e visível
   se o cliente da API mandar payload malformado (coluna faltando, NaN, tipo
   errado, ordem trocada).

2. **`PhysicalBoundsClipper`** — clipa cada feature aos limites físicos
   plausíveis (pH ∈ [0,14], N/P/K ≥ 0 etc.). Defensivo contra leituras de
   sensor absurdas em produção. **No treino, é no-op** (todos os valores já
   estão dentro dos limites).

3. **`NutrientRatioEngineer`** — adiciona 4 features derivadas: `NP_ratio`,
   `NK_ratio`, `PK_ratio`, `NPK_sum`. As razões NPK são uma métrica
   agronômica clássica — capturam o **equilíbrio** de nutrientes, informação
   complementar aos valores absolutos.

4. **`StandardScaler`** (via `ColumnTransformer`) — padronização (μ=0, σ=1)
   das 11 features (7 originais + 4 engenheiradas). Escolhi padronização
   porque (i) cobre o maior número de famílias de modelos, (ii) as
   distribuições já são razoavelmente normais, e (iii) não há outliers
   ruidosos justificando `RobustScaler`.

Entrada: 7 features × N amostras → Saída: **11 features × N amostras**

In [ ]:
pipeline = build_pipeline()
pipeline

In [ ]:
X, y = split_xy(df)
pipeline.fit(X, y)

transformed = pipeline.transform(X)
print(f"Input:  {X.shape}")
print(f"Output: {transformed.shape}")

stats = pd.DataFrame(transformed, columns=FINAL_FEATURES).describe().T[["mean","std","min","max"]]
stats.round(3)

## 9. Round-trip de serialização

In [ ]:
import joblib

PIPELINE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, PIPELINE_PATH)
print(f"Salvo em: {PIPELINE_PATH}")

reloaded = joblib.load(PIPELINE_PATH)
saida_original = pipeline.transform(X.head(10))
saida_recarregada = reloaded.transform(X.head(10))

np.testing.assert_allclose(saida_original, saida_recarregada)
print("Round-trip OK — pipeline serializa e recarrega com saída idêntica.")

### Validações do contrato em produção

In [ ]:
casos = [
    ("Falta coluna 'ph'", X.drop(columns=["ph"]).head(1)),
    ("NaN em N",          X.head(1).assign(N=np.nan)),
    ("Tipo errado em N",  X.head(1).assign(N="texto")),
    ("pH absurdo (25)",   X.head(1).assign(ph=25.0)),
]

for nome, payload in casos:
    try:
        out = reloaded.transform(payload)
        print(f"{nome:25s} -> ACEITO (clipado para limite). shape={out.shape}")
    except (ValueError, TypeError) as e:
        print(f"{nome:25s} -> BLOQUEADO: {e}")

- Casos 1, 2 e 3 (schema/NaN/tipo) → **bloqueados** com erro claro.
- Caso 4 (pH = 25) → **aceito mas clipado** para 14.0 antes de chegar no
  scaler. Comportamento desejado: sensor com defeito não derruba o
  serviço, mas o valor é normalizado.

A escolha entre bloquear e clipar foi deliberada: schema é contrato
**rígido** (cliente da API deve mandar payload correto); limites físicos
são **defesa em profundidade** contra sensores ruins.

## Resumo das decisões

| Tópico | Decisão | Razão |
|---|---|---|
| Imputação | nenhuma | 0 NaN |
| Duplicatas | nenhuma ação | 0 duplicatas |
| Outliers físicos | clipagem em prod | defesa contra sensor ruim, no-op no treino |
| "Outliers" IQR | manter | são assinatura biológica das classes |
| Multicolinearidade P-K (ρ=0.74) | manter ambas | ganho de interpretabilidade > custo |
| Balanço de classes | nada | já perfeitamente balanceado |
| Feature engineering | 4 razões NPK + soma | métricas agronômicas clássicas |
| Escalonamento | `StandardScaler` | melhor cobertura entre famílias de modelos |
| Validação em prod | `FeatureValidator` | falha cedo em schema ruim |
| Serialização | `joblib` | padrão sklearn, round-trip verificado |

**Artefato:** `artifacts/pipeline_preprocessador.pkl` — entrada de 7 features
brutas → saída de 11 features padronizadas. Pronto para consumo pela API
de inferência na próxima etapa.